In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import csv
import time
import chardet
import xmltodict

In [3]:
API_KEY = "qLwie1%2B4M48e8WU9A7li%2FvF9ETd1N%2F2Guqy8cOxUawVK5GjtsqmS8je1VK0CuqkyC3Q%2FMDWCjtybH4ZaYcD1GA%3D%3D"


SERVICE = "SpcdeInfoService/getHoliDeInfo"        # 행정동단위
file_name = "공휴일.csv"



BASE_URL = f"http://apis.data.go.kr/B090041/openapi/service/SpcdeInfoService/getHoliDeInfo?&ServiceKey={API_KEY}&numOfRows=40"

In [7]:
# 전체 갯수 확인


for year in range(2022, 2027) :

    url = f"{BASE_URL}&solYear={year}"

    # 요청
    res = requests.get(url)

    # 응답 XML 변환
    response = xmltodict.parse(res.text)

    # item 추출

    print(response)

    result_code = response["response"]["header"]["resultCode"]
    if result_code != "00":
        raise Exception(f"API 오류: {result_code}")

    items = response["response"]["body"]["items"]

    # DataFrame 변환
    df = pd.DataFrame(items)




{'response': {'header': {'resultCode': '00', 'resultMsg': 'NORMAL SERVICE.'}, 'body': {'items': {'item': [{'dateKind': '01', 'dateName': '1월1일', 'isHoliday': 'Y', 'locdate': '20260101', 'seq': '1'}, {'dateKind': '01', 'dateName': '설날', 'isHoliday': 'Y', 'locdate': '20260216', 'seq': '1'}, {'dateKind': '01', 'dateName': '설날', 'isHoliday': 'Y', 'locdate': '20260217', 'seq': '1'}, {'dateKind': '01', 'dateName': '설날', 'isHoliday': 'Y', 'locdate': '20260218', 'seq': '1'}, {'dateKind': '01', 'dateName': '삼일절', 'isHoliday': 'Y', 'locdate': '20260301', 'seq': '1'}, {'dateKind': '01', 'dateName': '대체공휴일(삼일절)', 'isHoliday': 'Y', 'locdate': '20260302', 'seq': '1'}, {'dateKind': '01', 'dateName': '노동절', 'isHoliday': 'Y', 'locdate': '20260501', 'seq': '2'}, {'dateKind': '01', 'dateName': '어린이날', 'isHoliday': 'Y', 'locdate': '20260505', 'seq': '2'}, {'dateKind': '01', 'dateName': '부처님오신날', 'isHoliday': 'Y', 'locdate': '20260524', 'seq': '1'}, {'dateKind': '01', 'dateName': '대체공휴일(부처님오신날)', 'isHolida

,item
0,"{'dateKind': '01', 'dateName': '1월1일', 'isHoli..."
1,"{'dateKind': '01', 'dateName': '설날', 'isHolida..."
2,"{'dateKind': '01', 'dateName': '설날', 'isHolida..."
3,"{'dateKind': '01', 'dateName': '설날', 'isHolida..."
4,"{'dateKind': '01', 'dateName': '삼일절', 'isHolid..."
5,"{'dateKind': '01', 'dateName': '대체공휴일(삼일절)', '..."
6,"{'dateKind': '01', 'dateName': '노동절', 'isHolid..."
7,"{'dateKind': '01', 'dateName': '어린이날', 'isHoli..."
8,"{'dateKind': '01', 'dateName': '부처님오신날', 'isHo..."
9,"{'dateKind': '01', 'dateName': '대체공휴일(부처님오신날)'..."


In [24]:
# 전체를 1000개 단위로 읽어오고 csv로 저장


is_first = True

step = 1000
ranges = [(i, min(i + step - 1, total_count)) for i in range(1, total_count + 1, step)]

for s, e in ranges:
    url = f"{BASE_URL}/{s}/{e}/"


    response = requests.get(url)

    data = response.json()

    body = data[SERVICE]

    result_code = body["RESULT"]["CODE"]
    if result_code != "INFO-000":
        raise Exception(f"API 오류: {body.get('RESULT')}")

    rows = body["row"]

    if not rows:
        continue

    df = pd.DataFrame(rows)

    df.to_csv(
        file_name,
        mode="a",
        index=False,
        header=is_first,
        encoding="utf-8-sig"
    )

    is_first = False

    del df


In [25]:
# 결측 데이터 확인

with open(file_name, 'rb') as f:
    result = chardet.detect(f.read(10000))

print(file_name, result)


df = pd.read_csv(file_name)

print(df.isnull().sum())


#df.to_csv(file_name, index=False, encoding='utf-8-sig')

서울시_행정동_지하철_승차수.csv {'encoding': 'UTF-8-SIG', 'confidence': 1.0, 'language': 'eo', 'mime_type': 'text/plain'}
CRTR_DD         0
DONG_ID         0
SBWY_PSNG       0
SBWY_PSNG_00    0
SBWY_PSNG_01    0
SBWY_PSNG_02    0
SBWY_PSNG_03    0
SBWY_PSNG_04    0
SBWY_PSNG_05    0
SBWY_PSNG_06    0
SBWY_PSNG_07    0
SBWY_PSNG_08    0
SBWY_PSNG_09    0
SBWY_PSNG_10    0
SBWY_PSNG_11    0
SBWY_PSNG_12    0
SBWY_PSNG_13    0
SBWY_PSNG_14    0
SBWY_PSNG_15    0
SBWY_PSNG_16    0
SBWY_PSNG_17    0
SBWY_PSNG_18    0
SBWY_PSNG_19    0
SBWY_PSNG_20    0
SBWY_PSNG_21    0
SBWY_PSNG_22    0
SBWY_PSNG_23    0
dtype: int64
